# Data Mining - Spark Data Lake Assignment
## 1. Data Lake Structure
First, we will set up the Data Lake structure locally and copy the dataset.

In [ ]:
import os
import shutil

# Create data lake directories
os.makedirs('data_lake/bronze', exist_ok=True)
os.makedirs('data_lake/silver', exist_ok=True)

# Copy the dataset to the bronze layer (assuming it's in the same directory)
if os.path.exists('dataset/sales.csv'):
    shutil.copy('dataset/sales.csv', 'data_lake/bronze/sales.csv')
else:
    print("Please ensure 'dataset/sales.csv' is available")

## 2. Load and Inspect with Spark
Install PySpark (if running in Colab), create a SparkSession, and read the bronze data.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, trim, lower, initcap, when, to_timestamp

spark = SparkSession.builder \
    .appName("DataLakeAssignment") \
    .getOrCreate()

# Load data from Bronze
df_bronze = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=True)

# Display the schema, a few rows, and the total record count
df_bronze.printSchema()
df_bronze.show(5)
print(f"Total Record Count: {df_bronze.count()}")

In [ ]:
# Identify duplicates, null values, inconsistencies

# 1. Duplicates
duplicates_count = df_bronze.count() - df_bronze.dropDuplicates().count()
print(f"Exact Duplicate Rows: {duplicates_count}")

# 2. Null values
print("Null values per column:")
df_bronze.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns]).show()

# 3. Invalid numerical values
if "quantity" in df_bronze.columns and "unit_price" in df_bronze.columns:
    invalid_qty = df_bronze.filter(col("quantity") <= 0).count()
    invalid_price = df_bronze.filter(col("unit_price") < 0).count()
    print(f"Invalid Quantity (<=0): {invalid_qty}")
    print(f"Invalid UnitPrice (<0): {invalid_price}")

# 4. Inconsistent/invalid dates can be checked by trying to cast to timestamp
if "order_date" in df_bronze.columns:
    invalid_dates = df_bronze.filter(to_timestamp(col("order_date")).isNull() & col("order_date").isNotNull()).count()
    print(f"Invalid order_date formats: {invalid_dates}")

## 3. Basic Preprocessing - Silver Layer
Perform the required preprocessing steps.

In [ ]:
df_silver = df_bronze
initial_count = df_silver.count()

# 1. Remove exact duplicate rows
df_silver = df_silver.dropDuplicates()

# 2. Handle missing values using a reasonable method
# Let's drop rows where critical IDs like order_id are null
if "order_id" in df_silver.columns:
    df_silver = df_silver.dropna(subset=["order_id"])

# 3. Trim extra spaces and standardize obvious capitalization inconsistencies in text fields
string_cols = [f.name for f in df_silver.schema.fields if f.dataType.typeName() == "string"]
for c in string_cols:
    df_silver = df_silver.withColumn(c, trim(col(c)))
    # Standardize capitalization for typical text fields
    if c in ["customer_name", "product", "category", "city", "state", "payment_method", "order_status"]:
        df_silver = df_silver.withColumn(c, initcap(lower(col(c))))

# 4. Apply basic validation to quantity, unit price, and discount percentage
if "quantity" in df_silver.columns:
    df_silver = df_silver.filter(col("quantity") > 0)
if "unit_price" in df_silver.columns:
    df_silver = df_silver.filter(col("unit_price") >= 0)
if "discount_percent" in df_silver.columns:
    # clip discount between 0 and 100
    df_silver = df_silver.withColumn("discount_percent", 
                                     when(col("discount_percent") < 0, 0)
                                     .when(col("discount_percent") > 100, 100)
                                     .otherwise(col("discount_percent")))

# 5. Convert valid dates to one consistent format and handle clearly invalid date values
if "order_date" in df_silver.columns:
    from pyspark.sql.functions import expr
    # Convert to timestamp/date
    df_silver = df_silver.withColumn("order_date", expr("try_cast(order_date as timestamp)"))
    # Drop records with invalid dates
    df_silver = df_silver.filter(col("order_date").isNotNull())

# 6. Preserve useful records where possible (done by targeted filtering)

# Save the resulting DataFrame inside the Silver folder in CSV format
silver_path = "data_lake/silver/preprocessed_sales"
df_silver.coalesce(1).write.mode("overwrite").csv(silver_path, header=True)
print(f"Data successfully saved to {silver_path}")

## 4. Verification
Show counts and sample the processed data.

In [ ]:
# Show the row count before and after preprocessing.
final_count = df_silver.count()
print(f"Row count before preprocessing: {initial_count}")
print(f"Row count after preprocessing: {final_count}")

# Read the Silver data back with Spark and display a few rows.
df_silver_read = spark.read.csv(silver_path, header=True, inferSchema=True)
df_silver_read.show(5)

# Briefly state which preprocessing operations were performed
print("\n--- Preprocessing Summary ---")
print("1. Exact duplicate rows were removed.")
print("2. Rows with missing 'order_id' were dropped.")
print("3. Extra spaces were trimmed from all text fields; capitalization was standardized (Initcap). ")
print("4. Filtered out records where quantity <= 0 or unit_price < 0.")
print("5. Discount percentages were bounded between 0 and 100.")
print("6. 'order_date' was converted to standard timestamp format, removing rows with completely invalid dates.")